In [1]:
!pip install -qU langchain langchain_openai langchain-core langchain-community langgraph psycopg[binary,pool]==3.2.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from langchain_core. prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import requests
#from langchain_community.utilities.sql_database import SQLDatabase
#from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [5]:
import os
with open("/content/api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

#with open("/content/ConexionSap.txt") as archivo:
 # uribd = archivo.read()

##Creación de Herramientas API

### PokeAPI

In [16]:
@tool
def extractorPokemon(name: str) -> str:
    """Con el nombre de un Pokémon retorna una cadena con sus tipos y dos habilidades, separadas por comas."""
    try:
        url = f"https://pokeapi.co/api/v2/pokemon/{name.lower()}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        # Extraer tipos
        types = [t["type"]["name"] for t in data.get("types", [])]
        # Extraer hasta dos habilidades
        abilities = [a["ability"]["name"] for a in data.get("abilities", [])[:2]]
        # Formatear como "tipo1,tipo2,habilidad1,habilidad2"
        return ",".join(types + abilities)
    except requests.HTTPError:
        return f"Error: Pokémon '{name}' no encontrado."
    except requests.RequestException:
        return "Error al conectar con PokeAPI"

### LinkedIn Scraper API

In [17]:
import json
import http.client
@tool
def extractorLinkedIn(linkedin_url: str) -> str:
    """Con la URL de un perfil de LinkedIn extrae solo el resumen {summary} y titular {headline} ."""
    try:
        conn = http.client.HTTPSConnection("linkedin-data-api.p.rapidapi.com")

        headers = {
            'x-rapidapi-key': "d7ab9f8d78msh4ee5cf021e44debp14954cjsn97468c6de7c9",
            'x-rapidapi-host': "linkedin-data-api.p.rapidapi.com"
        }

        encoded_url = linkedin_url.replace("/", "%2F")
        endpoint = f"/get-profile-data-by-url?url={encoded_url}"

        conn.request("GET", endpoint, headers=headers)
        res = conn.getresponse()

        if res.status != 200:
            return f"Error: Código {res.status} al consultar LinkedIn."

        data = json.loads(res.read().decode("utf-8"))

        summary = data.get("summary", "Resumen no disponible")
        headline = data.get("headline", "Titular no disponible")

        return f"{headline}, {summary}"

    except Exception as e:
        return f"Error al procesar la solicitud: {str(e)}"

### Amazon API

In [18]:
@tool
def extractorProductoAmazon(query: str) -> str:
    """Eres un asistente que busca productos en Amazon por palabra clave y devuelve las 3 primeras opciones de la página 1,
    indicando título, precio, rating, URL y más. Al final pregunta si quiere saber de otro producto."""
    try:
        url = (
            "https://real-time-amazon-data.p.rapidapi.com/search"
            f"?query={requests.utils.quote(query)}"
            "&page=1"
            "&country=US"
            "&sort_by=RELEVANCE"
            "&product_condition=ALL"
            "&is_prime=false"
            "&deals_and_discounts=NONE"
        )
        headers = {
            "x-rapidapi-key": "d7ab9f8d78msh4ee5cf021e44debp14954cjsn97468c6de7c9",
            "x-rapidapi-host": "real-time-amazon-data.p.rapidapi.com"
        }
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        data = resp.json()

        products = data.get("data", {}).get("products", [])[:3]
        if not products:
            return "No se encontraron productos para esa búsqueda."

        lines = []
        for i, p in enumerate(products, start=1):
            lines.append(
                f"{i}. Título: {p.get('product_title','N/A')}\n"
                f"   ASIN: {p.get('asin','N/A')}\n"
                f"   Precio: {p.get('product_price','N/A')} (Original: {p.get('product_original_price','N/A')})\n"
                f"   Rating: {p.get('product_star_rating','N/A')} ({p.get('product_num_ratings','N/A')} opiniones)\n"
                f"   URL: {p.get('product_url','N/A')}\n"
                f"   Foto: {p.get('product_photo','N/A')}\n"
                f"   Entrega: {p.get('delivery','N/A')}\n"
            )

        lines.append("¿Quieres buscar otro producto en Amazon?")
        return "\n".join(lines)

    except requests.HTTPError as e:
        return f"Error HTTP {resp.status_code} al consultar Amazon API."
    except requests.RequestException as e:
        return "Error al conectar con la Amazon API."


### API YOUTUBE MEDIA DONWLOADER

In [19]:
import urllib
@tool
def extractorAlbumes(cantante: str) -> str:
    """
    Eres un asistente que busca los últimos 3 playlists cuando te especifiquen el cantante;
    retorna solo los títulos (title) de cada playlist encontrados, separados por comas.
    """
    try:
        # Codificar el nombre para URL
        query = urllib.parse.quote(cantante)
        url = f"https://youtube-media-downloader.p.rapidapi.com/v2/search/playlists?keyword={query}&sortBy=relevance"

        # Cabeceras de RapidAPI
        headers = {
            "x-rapidapi-key": "d7ab9f8d78msh4ee5cf021e44debp14954cjsn97468c6de7c9",
            "x-rapidapi-host": "youtube-media-downloader.p.rapidapi.com"
        }

        # Llamada HTTP con requests
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()

        # Extraer hasta 5 títulos
        items = data.get("items", [])
        titles = [item.get("title", "<sin title>") for item in items[:5]]

        if not titles:
            return f"No se encontraron playlists para '{cantante}'."

        # Devolver en formato "título1,título2,..."
        return ",".join(titles)

    except requests.HTTPError:
        return f"Error: no se encontraron playlists para '{cantante}'."
    except requests.RequestException as e:
        return f"Error al conectar con la API: {e}"
    except ValueError:
        return "Error: la respuesta de la API no es un JSON válido."

### API EQUIPO DE FUTBOL

In [20]:
@tool
def posiciondejugador(futbolista: str) -> str:
      """con el nombre del futbolista retorname su posicion de juego en el campo de futbol"""
      try:
        url = "https://apifutbolista-ko6ygtybmq-wn.a.run.app/equipo?futbolista="+futbolista
        response = requests.get(url)
        if response.status_code == 200:
          return response.text
        else:
          return "Error encontrando equipo de futbol"
      except requests.exceptions.RequestException as e:
        return "Error encontrando equipo de futbol"


### API Tripulacion ONEPIECE

In [21]:
@tool
def obtenerTripulacion(personaje: str) -> str:
    """Dado el nombre de un personaje de One Piece, retorna el nombre de su tripulación pirata desde una API externa."""
    try:
        url = "https://apionepiece-ko6ygtybmq-wn.a.run.app/tripulacion?personaje="+personaje
        response = requests.get(url)
        if response.status_code == 200:
            return response.text
        else:
            return "Error encontrando tripulación del personaje"
    except requests.exceptions.RequestException:
        return "Error encontrando tripulación del personaje"

##AGENTE INTENGRANDO LAS HERRAMIENTAS

In [23]:
model = ChatOpenAI(verbose=True)
memory = MemorySaver()
#memory = ConversationBufferWindowMemory(
#    k=5,                # numero de mensajes a retener
 ##)
#construccion del prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """ Eres un asistente gentil, usa solo tus herramientas para responder las preguntas de los usuarios.
        Si no cuentas con una herramienta específica para resolver una pregunta, infórmalo claramente, indica que no puedes responder y evita realizar cualquier invocación innecesaria .
        """),

     ("human", "{messages}"),

     ]
)

tolkit = [extractorPokemon,extractorLinkedIn,extractorProductoAmazon,extractorAlbumes,posiciondejugador,obtenerTripulacion]
agent = create_react_agent(model, tolkit, checkpointer=memory, prompt=prompt)

In [24]:
config = {"configurable": {"thread_id": "abc126"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="¿a que tripulacion pertenece zoro?")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

¿a que tripulacion pertenece zoro?
================================== Ai Message ==================================
Tool Calls:
  obtenerTripulacion (call_arULlOJEePBFcj5K0IgCAftQ)
 Call ID: call_arULlOJEePBFcj5K0IgCAftQ
  Args:
    personaje: zoro
================================= Tool Message =================================
Name: obtenerTripulacion

Zoro pertenece a la tripulación pirata de los Sombrero de Paja, liderada por Monkey D. Luffy en el anime y manga One Piece.
================================== Ai Message ==================================

Zoro pertenece a la tripulación pirata de los Sombrero de Paja, liderada por Monkey D. Luffy en el anime y manga One Piece.


In [ ]:
config = {"configurable": {"thread_id": "abc126"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="¿en que posicion juega vinicius junior?")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

¿en que posicion juega vinicius junior?
================================== Ai Message ==================================
Tool Calls:
  buscarequipodefutbol (call_zeEeM8AZq0h8ERtLQdw5azne)
 Call ID: call_zeEeM8AZq0h8ERtLQdw5azne
  Args:
    futbolista: Vinicius Junior
  buscarequipodefutbol (call_DHA7pEsuqAG72PrNTWGE8Ovy)
 Call ID: call_DHA7pEsuqAG72PrNTWGE8Ovy
  Args:
    futbolista: Pedri
================================= Tool Message =================================
Name: buscarequipodefutbol

Mediocampista.
================================== Ai Message ==================================

Pedri juega en la posición de Mediocampista y Vinicius Junior juega en la posición de Delantero. ¿Hay algo más en lo que pueda ayudarte?


In [ ]:
config = {"configurable": {"thread_id": "abc126"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="que habilidad tiene Zapdos")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

que habilidad tiene Zapdos
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_5EZTLVTk8HiZsg3ar6Z5nkku)
 Call ID: call_5EZTLVTk8HiZsg3ar6Z5nkku
  Args:
    name: Zapdos
================================= Tool Message =================================
Name: extractorPokemon

electric,flying,pressure,static
================================== Ai Message ==================================

Zapdos tiene las siguientes habilidades:

- Tipo: Eléctrico, Volador
- Habilidades: Presión, Electrostática


In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="y de linkin park? ")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

y de linkin park? 
================================== Ai Message ==================================
Tool Calls:
  extractorAlbumes (call_btF1PXJ7IXsgAs4HWGIel7CH)
 Call ID: call_btF1PXJ7IXsgAs4HWGIel7CH
  Args:
    cantante: Linkin Park
================================= Tool Message =================================
Name: extractorAlbumes

playlist, Linkin Park - From Zero (Full Album)
playlist, Linkin Park - Hybrid Theory (Full Album)
playlist, Linkin Park - Top Tracks
playlist, Linkin Park Greatest Hits
playlist, Linkin Park - One More Light (Full Album)
playlist, Linkin Park - Meteora (Full Album)
playlist, Linkin Park - Live In Texas (Full Album)
playlist, Linkin Park - Reanimation (Full Album)
playlist, Youtube mix Linkin Park
playlist, Linkin Park - Hybrid Theory (Full Album)
playlist, Linkin Park From Zero (Deluxe Edition) | FULL ALBUM 2025
playlist, linkin park greatest hits
playlist, Linkin Park 

In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="y que habilidades tiene snorlax")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

y que habilidades tiene snorlax
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_eamlWzm6qNBdRjfClkwDbUsI)
 Call ID: call_eamlWzm6qNBdRjfClkwDbUsI
  Args:
    name: snorlax
================================= Tool Message =================================
Name: extractorPokemon

normal,immunity,thick-fat
================================== Ai Message ==================================

Snorlax tiene las siguientes habilidades: Normal, Inmunidad y Grasa sólida (Thick Fat).


In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="y de snoop dog? ")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

y de snoop dog? 
================================== Ai Message ==================================
Tool Calls:
  extractorAlbumes (call_mOgS6XB9hjKxI1ZiJ74OQbSW)
 Call ID: call_mOgS6XB9hjKxI1ZiJ74OQbSW
  Args:
    cantante: Snoop Dog
================================= Tool Message =================================
Name: extractorAlbumes

Snoop dogg tha last meal full album,Snoop Dogg - No Limit Top Dogg (1999),All Snoop Dogg Albums,Snoop Dogg murder was the case full album,Snoop Dogg - Missionary Album 2024
================================== Ai Message ==================================

Los álbumes de Snoop Dogg son:

1. Snoop Dogg - Tha Last Meal (Full Album)
2. Snoop Dogg - No Limit Top Dogg (1999)
3. All Snoop Dogg Albums
4. Snoop Dogg - Murder Was the Case (Full Album)
5. Snoop Dogg - Missionary Album 2024

¿Hay algo más en lo que pueda ayudarte?


In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="quiero comprar un celular cual recomiendas")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

quiero comprar un celular cual recomiendas
================================== Ai Message ==================================
Tool Calls:
  extractorProductoAmazon (call_Qhx0SCSm0FYNHq3NfsW0AWrB)
 Call ID: call_Qhx0SCSm0FYNHq3NfsW0AWrB
  Args:
    query: celular
================================= Tool Message =================================
Name: extractorProductoAmazon

1. Título: Tracfone | Motorola Moto g Play 2024 | Locked | 64GB | 5000mAh Battery | 50MP Quad Pixel Camera | 6.5-in. HD+ 90Hz Display | Sapphire Blue
   ASIN: B0CTW8TXGH
   Precio: $39.88 (Original: $49.99)
   Rating: 4.3 (661 opiniones)
   URL: https://www.amazon.com/dp/B0CTW8TXGH
   Foto: https://m.media-amazon.com/images/I/71CxUvG46rL._AC_UY654_FMwebp_QL65_.jpg
   Entrega: FREE delivery Wed, May 7 Or fastest delivery Tomorrow, May 3

2. Título: SAMSUNG Galaxy S25 Ultra Cell Phone, 256GB AI Smartphone, Unlocked Android, AI Camera, Fast P

In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="si quisiera saber delaptops")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

si quisiera saber delaptops
================================== Ai Message ==================================
Tool Calls:
  extractorProductoAmazon (call_jCYG67BsJgJBtOOdH7GphhrP)
 Call ID: call_jCYG67BsJgJBtOOdH7GphhrP
  Args:
    query: laptops
================================= Tool Message =================================
Name: extractorProductoAmazon

1. Título: Acer Aspire 3 A315-24P-R7VH Slim Laptop | 15.6&quot; Full HD IPS Display | AMD Ryzen 3 7320U Quad-Core Processor | AMD Radeon Graphics | 8GB LPDDR5 | 128GB NVMe SSD | Wi-Fi 6 | Windows 11 Home in S Mode
   ASIN: B0BS4BP8FB
   Precio: $299.99 (Original: $349.99)
   Rating: 4.2 (4247 opiniones)
   URL: https://www.amazon.com/dp/B0BS4BP8FB
   Foto: https://m.media-amazon.com/images/I/61gKkYQn6lL._AC_UY654_FMwebp_QL65_.jpg
   Entrega: FREE delivery Tue, May 6 Or fastest delivery Overnight 4 AM - 8 AM

2. Título: HP 14 Laptop, Intel Celeron N4020, 

In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="conoces el pokemon ditto?")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

conoces el pokemon ditto?
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_XbGoRcnfBcR1Rk7buHTazr8E)
 Call ID: call_XbGoRcnfBcR1Rk7buHTazr8E
  Args:
    name: ditto
================================= Tool Message =================================
Name: extractorPokemon

normal,limber,imposter
================================== Ai Message ==================================

¡Sí! Conozco al Pokémon Ditto. 
Ditto es de tipo Normal y posee las habilidades Limber e Imposter.


In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="y charmander?")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

y charmander?
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_muXl5KKYNeoN5ZCZOGvW4gPo)
 Call ID: call_muXl5KKYNeoN5ZCZOGvW4gPo
  Args:
    name: charmander
================================= Tool Message =================================
Name: extractorPokemon

fire,blaze,solar-power
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_iHsrcDiK2EaUsYy32SD1VBU9)
 Call ID: call_iHsrcDiK2EaUsYy32SD1VBU9
  Args:
    name: ditto
  extractorPokemon (call_3xWfc6ko0tId4vvdJkZNfVcT)
 Call ID: call_3xWfc6ko0tId4vvdJkZNfVcT
  Args:
    name: charmander
================================= Tool Message =================================
Name: extractorPokemon

fire,blaze,solar-power
================================== Ai Message ==================================

¡Sí! Conozco al Pokémon D

In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="y que me puedes decir de pikachu?")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

y que me puedes decir de pikachu?
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_WlZ7ZYUvEv0bcfMLXgfQ1qAw)
 Call ID: call_WlZ7ZYUvEv0bcfMLXgfQ1qAw
  Args:
    name: pikachu
  extractorPokemon (call_3H04kqEPTVR8gfXydA8mWkzA)
 Call ID: call_3H04kqEPTVR8gfXydA8mWkzA
  Args:
    name: pikachu
================================= Tool Message =================================
Name: extractorPokemon

electric,static,lightning-rod
================================== Ai Message ==================================

¡Claro! Aquí tienes la información que solicita sobre algunos Pokémon:

- Ditto es de tipo Normal y posee las habilidades Limber e Imposter.
- Charmander es de tipo Fuego y tiene las habilidades Blaze y Solar Power.
- Pikachu es de tipo Eléctrico y tiene las habilidades Static y Lightning Rod.


In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
response = agent.invoke({"messages": [HumanMessage(content="que sabes de Mewtwo")]}, config=config)
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Mewtwo es un Pokémon de tipo Psychic con las habilidades Pressure y Unnerve.


In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="que sabes de onix")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

que sabes de onix
================================== Ai Message ==================================
Tool Calls:
  extractorPokemon (call_eX3e2QRLsdZBEhfgYFDBdcAw)
 Call ID: call_eX3e2QRLsdZBEhfgYFDBdcAw
  Args:
    name: onix
  extractorPokemon (call_gLDK8VLi0mvEFIrZVTnN9wSd)
 Call ID: call_gLDK8VLi0mvEFIrZVTnN9wSd
  Args:
    name: onix
================================= Tool Message =================================
Name: extractorPokemon

rock,ground,rock-head,sturdy
================================== Ai Message ==================================

¡Claro! Aquí tienes la información sobre algunos Pokémon:

- Ditto es de tipo Normal y posee las habilidades Limber e Imposter.
- Charmander es de tipo Fuego y tiene las habilidades Blaze y Solar Power.
- Pikachu es de tipo Eléctrico y tiene las habilidades Static y Lightning Rod.
- Onix es de tipo Roca/Tierra y tiene las habilidades Rock Head y Sturdy. ¿En qué

In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="dame los datos de hector-cardenas-camacho-197101169")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

dame los datos de hector-cardenas-camacho-197101169
================================== Ai Message ==================================
Tool Calls:
  extractorLinkedIn (call_rSuR7kj3ZiyyVN4vTzJbsnL0)
 Call ID: call_rSuR7kj3ZiyyVN4vTzJbsnL0
  Args:
    linkedin_url: https://www.linkedin.com/in/hector-cardenas-camacho-197101169
================================= Tool Message =================================
Name: extractorLinkedIn

Data Analyst - TI | Programmer | Developer in Power BI, SAP Datasphere and SAP Analytics Cloud | SAP ABAP Developer Reports, Ingeniero Industrial con una fuerte pasión por el mundo de los datos y la programación. Durante los últimos 4 años, he estado inmerso en proyectos tecnológicos en el área de Sistemas/TI. Poseo especialización en Análisis de Datos y Business Intelligence, utilizando herramientas como SQL y Power BI, así como en Machine Learning y Deep Learning con Python. Ademá

In [26]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="dame informacion de este perfil https://www.linkedin.com/in/mcotrina/")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

dame informacion de este perfil https://www.linkedin.com/in/mcotrina/
================================== Ai Message ==================================
Tool Calls:
  extractorLinkedIn (call_8KVzXiNJRNQPlOgquMdRmSTi)
 Call ID: call_8KVzXiNJRNQPlOgquMdRmSTi
  Args:
    linkedin_url: https://www.linkedin.com/in/mcotrina/
================================= Tool Message =================================
Name: extractorLinkedIn

Solutions Architect for the Data & AI Practice at Indra | Master’s degree in Data Science | Google Cloud Certified | AI/ML/DL & Gen AI Specialist | LangChain RAG, Big Data & Cloud Instructor | Driving Data Innovation, As a fervent Information Technology strategist with over a decade dedicated to excellence and innovation in the IT sector, I specialize in turning technological challenges into business opportunities, offering solutions that not only meet current needs but also pave the way 

In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="dame informacion de este perfil https://www.linkedin.com/in/mcotrina/")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

dame informacion de este perfil https://www.linkedin.com/in/mcotrina/
================================== Ai Message ==================================
Tool Calls:
  extractorLinkedIn (call_NacYO0yHMuEj8Idef3m8JNSZ)
 Call ID: call_NacYO0yHMuEj8Idef3m8JNSZ
  Args:
    linkedin_url: https://www.linkedin.com/in/mcotrina/
================================= Tool Message =================================
Name: extractorLinkedIn

Solutions Architect for the Data & AI Practice at Indra | Master’s degree in Data Science | Google Cloud Certified | AI/ML/DL & Gen AI Specialist | LangChain RAG, Big Data & Cloud Instructor | Driving Data Innovation, As a fervent Information Technology strategist with over a decade dedicated to excellence and innovation in the IT sector, I specialize in turning technological challenges into business opportunities, offering solutions that not only meet current needs but also pave the way 

In [ ]:
config = {"configurable": {"thread_id": "abc124"}}
for step in agent.stream(
    {"messages": [HumanMessage(content="dame informacion de este perfil https://www.linkedin.com/in/mcperezes/")]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

dame informacion de este perfil https://www.linkedin.com/in/mcperezes/
================================== Ai Message ==================================
Tool Calls:
  extractorLinkedIn (call_tozveWjd8UdGDYoqJXcryPCp)
 Call ID: call_tozveWjd8UdGDYoqJXcryPCp
  Args:
    linkedin_url: https://www.linkedin.com/in/mcperezes/
================================= Tool Message =================================
Name: extractorLinkedIn

Software Developer  (DevOps |  Azure | Next Js | React Js | HTML 5 | CSS |  Redux | Typescript | Javascript ), Hola, soy María Claudia, pero prefiero que me llamen Macu.

Soy desarrolladora Frontend con conocimientos en Backend y cultura DevOps. Me apasiona la tecnología y siempre estoy explorando nuevas formas de aprender e innovar.

Como toda buena programadora, amo el café (tanto que hice un curso de barismo ☕). También disfruto escribir y he combinado esa pasión con la programación,

In [14]:
# 1) Instala Gradio
!pip install -qU gradio

# 2) Importa Gradio y los mensajes de LangChain
import gradio as gr
from langchain_core.messages import HumanMessage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.9 MB/s eta 0:00:00


In [25]:
# 3) Envuelve tu agente en una función
def run_agent(prompt: str) -> str:
    """Recibe el texto del usuario y devuelve la respuesta del agente."""
    # Llamas a tu agent ya instanciado más arriba
    respuesta = ""
    for step in agent.stream(
        {"messages": [HumanMessage(content=prompt)]},
        config,                  # asume que tienes `config` definido
        stream_mode="values",
    ):
        # cada paso es un dict con {"messages": [...]}
        respuesta = step["messages"][-1].content
    return respuesta

# 4) Crea la interfaz Gradio
iface = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(lines=2, placeholder="Escribe tu pregunta aquí..."),
    outputs=gr.Textbox(label="Respuesta"),
    title="Agente Conversacional de IA Multiherramientas",
    description="Pregunta a tu agente que integra PokeAPI, LinkedIn, Amazon, OnePiece…"
)

# 5) Lanza el servidor (en Colab te dará una URL pública)
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a1b6e9bf212f7715b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
